In [3]:
import numpy as np
import numpy.lib.recfunctions as recfun
from pathlib import Path
from plyfile import PlyData, PlyElement
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import open3d as o3d
import pandas as pd

from matplotlib import cm
import copy

import random
import os
import gc

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchmetrics.classification import MulticlassF1Score, MulticlassPrecision, MulticlassRecall, MulticlassJaccardIndex, BinaryAccuracy, BinaryMatthewsCorrCoef, MulticlassConfusionMatrix

import torchsummary
import time

In [4]:
DOWNSAMPLING_GRID = {
    'voxel': [{'resolution_percentage': 0.01}, {'resolution_percentage': 0.02}, {'resolution_percentage': 0.03}, {'resolution_percentage': 0.04}, {'resolution_percentage': 0.05}],
    'fps': [{'retention_rate': 0.05}, {'retention_rate': 0.10}, {'retention_rate': 0.20}],
    'poisson': [{'radius': 1}, {'radius': 2}, {'radius': 3}],
    'random': [{'retention_rate': 0.05}, {'retention_rate': 0.10}, {'retention_rate': 0.20}],
    'uniform': [{'k_step': 5}, {'k_step': 10}, {'k_step': 20}]
}

NORM_STRATEGIES = ['val', 'train_ds', 'separate', 'train_full']
LEARNING_RATES = [0.01, 0.005, 0.001]
BATCH_SIZES = [512, 1024]

Path('../data/processed_data').mkdir(parents=True, exist_ok=True)
Path('../data/results').mkdir(parents=True, exist_ok=True)

In [5]:
def set_seed(seed=42):
    # Python & OS
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # NumPy
    np.random.seed(seed)
    
    # PyTorch CPU
    torch.manual_seed(seed)
    
    # PyTorch GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        
    # Backend CUDA (CuDNN)
    #torch.backends.cudnn.deterministic = True
    #torch.backends.cudnn.benchmark = True
    
    #print(f"Global seed fixed on {seed} (Deterministic CuDNN activated)")
    print(f"Global seed fixed on {seed}")

In [6]:
set_seed()

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

Global seed fixed on 42


In [7]:
def load_dataset_ply_lb(dataset='Train', base_dir='../data/Challenge-ABC'):
    base_path = Path(base_dir) / dataset
    if not base_path.exists():
        print(f"Error: Dataset folder {base_path} not found.")
        return None, None, None

    ply_dir = base_path / 'ply'
    lb_dir = base_path / 'lb'

    ply_files = list(ply_dir.glob('*.ply'))
    total_files = len(ply_files)

    points_list = []
    labels_list = []
    file_ids = []

    for i, ply_file in enumerate(ply_files, 1):
        file_id = ply_file.stem
        lb_file = lb_dir / f"{file_id}.lb"

        # Check that the labels file exists
        if not lb_file.exists():
            print(f"Error: Didn't find labels file for {file_id}.ply -> ignored.")
            continue

        # Load labels
        labels = np.loadtxt(lb_file, dtype='int')
        
        # Load point cloud using Open3D
        pcd = o3d.io.read_point_cloud(str(ply_file))
        points = np.asarray(pcd.points)

        # Check dimension consistency
        if len(points) != len(labels):
            print(f"Error: Dimension error with {file_id} : {len(points)} points vs {len(labels)} labels -> ignored.")
            continue

        # Append to lists to maintain object separation
        points_list.append(points)
        labels_list.append(labels)
        file_ids.append(file_id)

        # Progress tracking
        if i % 20 == 0:
            print(f"Loading : {i}/{total_files} files...")
            
    print(f"Successfully loaded {len(file_ids)}/{total_files} files.")

    return points_list, labels_list, file_ids

In [8]:
def downsample_non_edges(points, labels, method, display=False, **kwargs):
    """
    Downsamples the non-edge points (label 0) of a point cloud while preserving all edge points (label 1).
    
    Parameters:
    - points: (N, 3) numpy array of spatial coordinates.
    - labels: (N,) numpy array of binary labels (1 for edge, 0 for non-edge).
    - method: String specifying the down-sampling algorithm ('voxel', 'fps', 'poisson', 'random', 'uniform').
    - display: Boolean. If True, renders the intermediate and final point clouds via plotly.
    - **kwargs: Method-specific parameters:
        - voxel: 'resolution_percentage' (e.g., 0.02 for 2% of max dimension)
        - fps: 'num_points' (int)
        - poisson: 'radius' (float)
        - random: 'num_points' (int)
        - uniform: 'k_step' (int)
    
    Returns:
    - final_global_idx: (M,) numpy array of the globally sorted indices to keep.
    """
    # 1. Initialize structure and extract global masks
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    
    global_edge_idx = np.where(labels == 1)[0]
    global_non_edge_idx = np.where(labels != 1)[0] # Security; using != 1 to catch any unclassified points
    
    non_edge_pcd = pcd.select_by_index(global_non_edge_idx)
    
    # 2. Routing logic for the selected algorithm
    method = method.lower()
    if method == 'voxel':
        if 'resolution_percentage' not in kwargs:
            raise ValueError("Method 'voxel' requires 'resolution_percentage'")
        
        # Calculate bounds on the full point cloud to maintain consistent relative scale
        min_bound = points.min(axis=0)
        max_bound = points.max(axis=0)
        max_dim = np.max(max_bound - min_bound)
        dynamic_voxel_size = max_dim * kwargs['resolution_percentage']
        
        _, _, trace = non_edge_pcd.voxel_down_sample_and_trace(
            voxel_size=dynamic_voxel_size, min_bound=min_bound, max_bound=max_bound
        )
        rel_idx = np.array([v[np.random.randint(0, len(v))] for v in trace])
        selected_non_edge_global_idx = global_non_edge_idx[np.sort(rel_idx)]
        
    elif method == 'fps':
        if 'retention_rate' in kwargs:
            target_pts = max(1, int(len(global_non_edge_idx) * kwargs['retention_rate']))
        elif 'num_points' in kwargs:
            target_pts = min(kwargs['num_points'], len(global_non_edge_idx))
        else:
            raise ValueError("Method 'fps' requires 'retention_rate' or 'num_points'")
        
        fps_pcd = non_edge_pcd.farthest_point_down_sample(target_pts)
        tree = o3d.geometry.KDTreeFlann(non_edge_pcd)
        rel_idx = np.zeros(target_pts, dtype=int)
        
        for i, pt in enumerate(fps_pcd.points):
            rel_idx[i] = tree.search_knn_vector_3d(pt, 1)[1][0]
            
        selected_non_edge_global_idx = global_non_edge_idx[rel_idx]

    elif method == 'poisson':
        if 'radius' not in kwargs:
            raise ValueError("Method 'poisson' requires 'radius'")
        tree = o3d.geometry.KDTreeFlann(non_edge_pcd)
        pts = np.asarray(non_edge_pcd.points)
        num_pts = len(pts)
        
        active_mask = np.ones(num_pts, dtype=bool)
        shuffled_idx = np.random.permutation(num_pts)
        rel_idx = []
        
        for idx in shuffled_idx:
            if not active_mask[idx]: continue
            rel_idx.append(idx)
            _, neighbors_idx, _ = tree.search_radius_vector_3d(pts[idx], kwargs['radius'])
            active_mask[np.asarray(neighbors_idx)] = False
            
        selected_non_edge_global_idx = global_non_edge_idx[rel_idx]

    elif method == 'random':
        if 'retention_rate' in kwargs:
            target_pts = max(1, int(len(global_non_edge_idx) * kwargs['retention_rate']))
        elif 'num_points' in kwargs:
            target_pts = min(kwargs['num_points'], len(global_non_edge_idx))
        else:
            raise ValueError("Method 'random' requires 'retention_rate' or 'num_points'")
        
        selected_non_edge_global_idx = np.random.choice(global_non_edge_idx, size=target_pts, replace=False)
        selected_non_edge_global_idx = np.sort(selected_non_edge_global_idx)

    elif method == 'uniform':
        if 'k_step' not in kwargs:
            raise ValueError("Method 'uniform' requires 'k_step'")
        selected_non_edge_global_idx = global_non_edge_idx[::kwargs['k_step']]
    else:
        raise ValueError(f"Unknown method '{method}'")

    # 3. Final Reintegration
    final_global_idx = np.concatenate([global_edge_idx, selected_non_edge_global_idx])
    final_global_idx = np.sort(final_global_idx)

    # 4. Diagnostics and Visualization
    if display:
        print(f"\n--- Downsampling Report ({method.upper()}) ---")
        print(f"Original Cloud Size : {len(points)}")
        print(f"Edge Points Kept    : {len(global_edge_idx)}")
        print(f"Non-Edge Downsampled: {len(selected_non_edge_global_idx)}")
        print(f"Final Cloud Size    : {len(final_global_idx)}")
        
        if method == 'voxel':
            print(f"Computed Voxel Size : {dynamic_voxel_size:.4f}")
        
        inter_pcd = pcd.select_by_index(selected_non_edge_global_idx)
        res_pcd = pcd.select_by_index(final_global_idx)
        
        res_pcd.paint_uniform_color([0, 0, 1])
        inter_pcd.paint_uniform_color([0, 0, 1])
        edge_colors = np.zeros((len(final_global_idx), 3))
        edge_colors[:] = [0, 0, 1]
        
        edge_mask_in_final = np.isin(final_global_idx, global_edge_idx)
        edge_colors[edge_mask_in_final] = [1, 0, 0] 
        res_pcd.colors = o3d.utility.Vector3dVector(edge_colors)
        

        o3d.visualization.draw_plotly([inter_pcd])
        o3d.visualization.draw_plotly([res_pcd])

    return final_global_idx

In [9]:
def build_downsampled_features(points_list, labels_list, file_ids, method, dataset='Train', base_dir='../data/Challenge-ABC', **kwargs):
    ssm_dir = Path(base_dir) / dataset / 'SSM_Challenge-ABC'
    
    X_list = []
    y_list = []
    
    total_files = len(file_ids)
    print(f"Starting feature extraction using '{method}' downsampling...")

    for i in range(total_files):
        file_id = file_ids[i]
        points = points_list[i]
        labels = labels_list[i]
        
        # Compute the global indices to keep for this specific model
        final_idx = downsample_non_edges(points, labels, method=method, display=False, **kwargs)
        
        # Load the corresponding .ssm file
        ssm_file = ssm_dir / f"{file_id}.ssm"
        if not ssm_file.exists():
            print(f"Error: {ssm_file.name} not found -> ignored.")
            continue
        features = np.loadtxt(ssm_file, skiprows=5, dtype='float32').reshape(-1, 20, 16)
        
        # Dimension consistency check
        if len(features) != len(points):
            print(f"Error: Dimension mismatch in {file_id}.ssm ({len(features)} features vs {len(points)} points) -> ignored.")
            continue
            
        # Slice the arrays using the downsampled indices
        filtered_features = features[final_idx]
        filtered_labels = labels[final_idx]
        
        X_list.append(filtered_features)
        y_list.append(filtered_labels)
        
        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{total_files} feature files...")
            
    print(f"Feature extraction complete. Processed {len(X_list)} valid files.")
    
    return X_list, y_list

In [10]:
def load_full_dataset(dataset='Train'):
    # 1) Path Definition
    # base directory for the chosen dataset
    base_path = Path('../data/Challenge-ABC') / dataset
    if not base_path.exists(): # verification
        print(f"Error: Dataset folder {base_path} not found, check 'dataset' argument.")
        return None
    ssm_dir = base_path / 'SSM_Challenge-ABC'
    lb_dir = base_path / 'lb'

    X_list = []
    y_list = []

    ssm_files = list(ssm_dir.glob('*.ssm'))
    total_files = len(ssm_files)

    for i, ssm_file in enumerate(ssm_files, 1):
        file_id = ssm_file.stem
        lb_file = lb_dir / f"{file_id}.lb"

        # check that the labels file exists
        if not lb_file.exists():
            print(f"Error: Didn't find labels file for {file_id}.ply -> ignored.")
            continue

        features = np.loadtxt(ssm_file, skiprows=5, dtype='float32').reshape(-1, 20, 16)
        labels = np.loadtxt(lb_file, dtype='int')

        if len(labels) == len(features):
            X_list.append(features)
            y_list.append(labels)
        else:
            print(f"Error: Dimension error for {file_id} -> ignored.")

        if i % 20 == 0:
            print(f"Loading : {i}/{total_files} files...")

    X = np.vstack(X_list)
    y = np.concatenate(y_list)    
    
    return X, y

In [11]:
class MLP(nn.Module):
    def __init__(self, input_dim=320, num_hidden_1=64, num_hidden_2=32):
        super(MLP, self).__init__()
        self.layer_1 = torch.nn.Linear(input_dim, num_hidden_1)
        self.layer_2 = torch.nn.Linear(num_hidden_1, num_hidden_2)
        self.layer_3 = torch.nn.Linear(num_hidden_2, 2)
        self.num_hidden_1 = num_hidden_1
        self.num_hidden_2 = num_hidden_2

    def forward(self, x):
        # Flatten the input from [batch_size, 20, 16] to [batch_size, 320]
        x = x.view(x.size(0), -1)

        # 1st layer
        out = self.layer_1(x)
        #out = torch.tanh(out) 
        out = torch.relu(out)
        
        # 2nd layer
        out = self.layer_2(out)
        out = torch.relu(out)

        # 3rd/output layer
        out = self.layer_3(out)
        return out

In [12]:
# Train Function
def train_mlp_model(X_train, y_train, X_val, y_val, nb_epochs=50, batch_size=1024, lr=0.01, device='cpu', display_metrics=False, seed=42, patience=10):
    set_seed(seed)

    input_dim = X_train.size(1) 
    
    start_time = time.time()
    print(f"Training on device: {device} (Seed: {seed}) | Features in input: {input_dim}")
    model = MLP(input_dim=input_dim).to(device)
    X_train_gpu = X_train.to(device)
    y_train_gpu = y_train.to(device)
    X_val_gpu = X_val.to(device)
    y_val_gpu = y_val.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    #optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    
    # --- Early Stopping and Best Model SEtup --
    best_val_loss = float('inf')
    nb_epochs_no_improvement = 0
    best_epoch = 1
    best_model_weights = copy.deepcopy(model.state_dict) 

    # --- Metrics ---
    # Dictionnary to store the complete history
    history = {
        'train_loss_epoch': [], 'train_loss_step': [], 'val_loss': [], 
        'val_acc': [], 'val_mcc': [],
        'val_precision_0': [], 'val_precision_1': [], 
        'val_recall_0': [], 'val_recall_1': [], 
        'val_f1_0': [], 'val_f1_1': [], 
        'val_iou_0': [], 'val_iou_1': [],
        'val_tp':[], 'val_tn':[], 'val_fp':[], 'val_fn':[]
    }


    # use Multiclass with average='none' to get a tensor with [score_class_0, score_class_1]
    f1_metric = MulticlassF1Score(num_classes=2, average='none').to(device)
    precision_metric = MulticlassPrecision(num_classes=2, average='none').to(device)
    recall_metric = MulticlassRecall(num_classes=2, average='none').to(device)
    iou_metric = MulticlassJaccardIndex(num_classes=2, average='none').to(device)
    mcc_metric = BinaryMatthewsCorrCoef().to(device)
    acc_metric = BinaryAccuracy().to(device)
    conf_matrix_metric = MulticlassConfusionMatrix(num_classes=2).to(device)

    
    print("Start of training...")
    
    for epoch in range(1, nb_epochs + 1):

        # --- Training Phase ---
        model.train()
        running_train_loss = 0.0
        
        # Create random permutation of indexes to shuffle the dataset at each epoch withour copying the whole dataset 
        idx = torch.randperm(X_train_gpu.size(0), device=device)

        # Loop over the dataset but instead of copying and shuffling the whole dataset, we work on the indexes 
        # so we slice only the indexes and pick the data directly from the original tensor based on the shuffled indexes
        for i in range(0, X_train_gpu.size(0), batch_size):
            # extract only the indexes for the current batch
            batch_idx = idx[i : i + batch_size]

            # pick the batch data from the data tensor 
            inputs = X_train_gpu[batch_idx]
            labels = y_train_gpu[batch_idx]

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            # added the loss at each step to the history for more detailed overview
            history['train_loss_step'].append(loss.item())
            running_train_loss += loss.item() * inputs.size(0)   
            
        epoch_train_loss = running_train_loss / X_train_gpu.size(0)
        history['train_loss_epoch'].append(epoch_train_loss)

        # --- Validation Phase ---
        model.eval()
        running_val_loss = 0.0

        f1_metric.reset()
        precision_metric.reset()
        recall_metric.reset()
        mcc_metric.reset()
        iou_metric.reset()
        acc_metric.reset()
        conf_matrix_metric.reset()

        with torch.no_grad():
            # same idea but without shuffling and no need to compute gradients
            for i in range(0, X_val_gpu.size(0), batch_size):
                inputs = X_val_gpu[i : i + batch_size]
                labels = y_val_gpu[i : i + batch_size]
      
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                running_val_loss += loss.item() * inputs.size(0)

                _, predicted = torch.max(outputs, 1)

                f1_metric.update(predicted, labels)
                precision_metric.update(predicted, labels)
                recall_metric.update(predicted, labels)
                mcc_metric.update(predicted, labels)
                iou_metric.update(predicted, labels)
                acc_metric.update(predicted, labels)
                conf_matrix_metric.update(predicted, labels)
                
        epoch_val_loss = running_val_loss / X_val_gpu.size(0)
        history['val_loss'].append(epoch_val_loss)
        
        
        # --- Compute Evaluation Metrics ---
        # now compute() returns a tensor of size 2 : [score_class_0, score_class_1]
        val_prec = precision_metric.compute()
        val_rec = recall_metric.compute()
        val_f1 = f1_metric.compute()
        val_iou = iou_metric.compute()
        conf_mat = conf_matrix_metric.compute()
        
        history['val_precision_0'].append(val_prec[0].item())
        history['val_precision_1'].append(val_prec[1].item())
        history['val_recall_0'].append(val_rec[0].item())
        history['val_recall_1'].append(val_rec[1].item())
        history['val_f1_0'].append(val_f1[0].item())
        history['val_f1_1'].append(val_f1[1].item())
        history['val_iou_0'].append(val_iou[0].item())
        history['val_iou_1'].append(val_iou[1].item())

        # global metrics computed as before
        val_mcc = mcc_metric.compute().item()
        val_acc = acc_metric.compute().item() * 100.0
        
        history['val_mcc'].append(val_mcc)
        history['val_acc'].append(val_acc)

        history['val_tn'].append(conf_mat[0, 0].item())
        history['val_fp'].append(conf_mat[0, 1].item())
        history['val_fn'].append(conf_mat[1, 0].item())
        history['val_tp'].append(conf_mat[1, 1].item())

        if display_metrics:
            print(f"Epoch [{epoch}/{nb_epochs}] | T.Loss: {epoch_train_loss:.4f} | V.Loss: {epoch_val_loss:.4f} | F1 (Class 1): {val_f1[1].item():.4f} | MCC: {val_mcc:.4f}")

        # --- Early Stopping Logic ---
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_epoch = epoch
            best_model_weights = copy.deepcopy(model.state_dict())
            nb_epochs_no_improvement = 0
        else:
            nb_epochs_no_improvement += 1
        
        if nb_epochs_no_improvement >= patience:
            print(f"\nEarly Stopping triggered at epoch {epoch}. Returning best weigths to now.")
            break

    history['best_epoch'] = best_epoch
    
    # --- Best Model ---
    print(f"Getting best model weights with val_loss = {best_val_loss:.4f}")
    model.load_state_dict(best_model_weights)

    # Free VRAM data 
    del X_train_gpu, y_train_gpu, X_val_gpu, y_val_gpu, f1_metric, precision_metric, recall_metric, mcc_metric, iou_metric, acc_metric, conf_matrix_metric
    torch.cuda.empty_cache()
    
    end_time = time.time()
    print(f"Training took {(end_time - start_time) / 60:.2f} minutes.")
    
    return history, model

In [13]:
def plot_metrics(metrics_history):
    fig, axs = plt.subplots(5, 2, figsize=(20, 25), gridspec_kw={'height_ratios': [1, 1, 1, 1, 1.5]})

    # Train Loss per step
    axs[0, 0].set_title("Train Loss (per step)")
    axs[0, 0].plot(metrics_history['train_loss_step'], color='blue', alpha=0.75, linewidth=0.75)
    axs[0, 0].set_xlabel("Iterations (Batches)")
    axs[0, 0].set_ylabel("Loss")
    axs[0, 0].grid(True, alpha=0.5)

    # Loss per Epoch Train and Val
    axs[0, 1].set_title("Loss (per Epoch)")
    axs[0, 1].plot(metrics_history['train_loss_epoch'], label="Train Loss", marker='o', color='blue')
    axs[0, 1].plot(metrics_history['val_loss'], label="Validation Loss", marker='o', color='red')
    axs[0, 1].set_xlabel("Epoch")
    axs[0, 1].set_ylabel("Loss")
    axs[0, 1].legend()
    axs[0, 1].grid(True, alpha=0.5)

    # Precision and Recall Class 1
    axs[1, 0].set_title("Precision & Recall (Class 1 - Validation)")
    axs[1, 0].plot(metrics_history['val_precision_1'], label='Precision', color='red')
    axs[1, 0].plot(metrics_history['val_recall_1'], label='Recall', color='green')
    axs[1, 0].set_xlabel("Epoch")
    axs[1, 0].set_ylabel("Score")
    axs[1, 0].legend()
    axs[1, 0].grid(True, alpha=0.5)

    # Precision and Recall Class 0
    axs[1, 1].set_title("Precision & Recall (Class 0 - Validation)")
    axs[1, 1].plot(metrics_history['val_precision_0'], label='Precision', color='red')
    axs[1, 1].plot(metrics_history['val_recall_0'], label='Recall', color='green')
    axs[1, 1].set_xlabel("Epoch")
    axs[1, 1].set_ylabel("Score")
    axs[1, 1].legend()
    axs[1, 1].grid(True, alpha=0.5)

    # F1-Score
    axs[2, 0].set_title("F1-Score per Class (Validation)")
    axs[2, 0].plot(metrics_history['val_f1_0'], label="F1 - Class 0", linestyle='--', color='blue')
    axs[2, 0].plot(metrics_history['val_f1_1'], label="F1 - Class 1", linewidth=2, color='red')
    axs[2, 0].set_xlabel("Epoch")
    axs[2, 0].set_ylabel("Score")
    axs[2, 0].legend()
    axs[2, 0].grid(True, alpha=0.5)

    # Global Metrics (MCC) & IoU
    axs[2, 1].set_title("MCC (Global) & IoU (per Class)")
    axs[2, 1].plot(metrics_history['val_mcc'], label="MCC (Global)", color='brown', marker='s')
    axs[2, 1].plot(metrics_history['val_iou_0'], label="IoU - Class 0", linestyle='--', color='blue')
    axs[2, 1].plot(metrics_history['val_iou_1'], label="IoU - Class 1", linewidth=2, color='red')
    axs[2, 1].set_xlabel("Epoch")
    axs[2, 1].set_ylabel("Score")
    axs[2, 1].legend()
    axs[2, 1].grid(True, alpha=0.5)

    # Class 1 Prediction tracking TP and FN 
    axs[3, 0].set_title("Edge Tracking: TP and FN")
    axs[3, 0].plot(metrics_history['val_tp'], label="TP = Correct Edges", color='green', marker='^')
    axs[3, 0].plot(metrics_history['val_fn'], label="FN = Missed Edges", color='red', marker='v')
    axs[3, 0].set_xlabel("Epoch")
    axs[3, 0].set_ylabel("Count")
    axs[3, 0].set_yscale('log')
    axs[3, 0].legend()
    axs[3, 0].grid(True, alpha=0.5)

    # Class 0 Prediction tracking TN and FP 
    axs[3, 1].set_title("Non-Edge Tracking: TN and FP")
    axs[3, 1].plot(metrics_history['val_tn'], label="TN = Correct Non-Edges", color='green', marker='^')
    axs[3, 1].plot(metrics_history['val_fp'], label="FP = Fake Edges", color='red', marker='v')
    axs[3, 1].set_xlabel("Epoch")
    axs[3, 1].set_ylabel("Count")
    axs[3, 1].set_yscale('log')
    axs[3, 1].legend()
    axs[3, 1].grid(True, alpha=0.5)

    # Best Epoch Data Extraction
    # Confusion Matrix of the best epoch
    best_epoch = metrics_history['best_epoch']
    best_idx = best_epoch - 1
    best_cm = np.array([
        [metrics_history['val_tn'][best_idx], metrics_history['val_fp'][best_idx]],
        [metrics_history['val_fn'][best_idx], metrics_history['val_tp'][best_idx]]
    ])

    hex_colors = ['#2ca02c', '#d62728', '#ff7f0e', '#1f77b4']
    color_map = ListedColormap(hex_colors) 
    color_indices = np.array([
        [0, 1], 
        [2, 3]  
    ])
    axs[4, 0].matshow(color_indices, cmap=color_map)
    
    axs[4, 0].set_title(f"Confusion Matrix - Best Epoch: {best_epoch}")
    axs[4, 0].set_xlabel("Predicted Label")
    axs[4, 0].set_ylabel("True Label")
    axs[4, 0].set_xticks([0, 1])
    axs[4, 0].set_yticks([0, 1])
    axs[4, 0].set_xticklabels(['Class 0', 'Class 1'])
    axs[4, 0].set_yticklabels(['Class 0', 'Class 1'])

    for (i, j), val in np.ndenumerate(best_cm):
        axs[4, 0].text(j, i, f"{int(val)}", ha='center', va='center', color='white', fontsize=16, fontweight='bold')

    # custom legend elements mapping the colors to the labels
    legend_elements = [
        Patch(facecolor=hex_colors[0], edgecolor='gray', label='TN (Correct Non-Edges)'),
        Patch(facecolor=hex_colors[1], edgecolor='gray', label='FP (Fake Edges)'),
        Patch(facecolor=hex_colors[2], edgecolor='gray', label='FN (Missed Edges)'),
        Patch(facecolor=hex_colors[3], edgecolor='gray', label='TP (Correct Edges)')
    ]

    axs[4, 0].legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1, 0.5), 
                     fontsize=10, frameon=True, edgecolor='gray')


    # Summary Text Box
    axs[4, 1].axis('off') # Hide axes
    summary_text = (
        f"BEST MODEL METRICS (Epoch {best_epoch})\n\n"
        f"Validation Loss :  {metrics_history['val_loss'][best_idx]:.4f}\n"
        f"Accuracy        :  {metrics_history['val_acc'][best_idx]:.2f}%\n"
        f"MCC (Global)    :  {metrics_history['val_mcc'][best_idx]:.4f}\n\n"
        f"-- CLASS 1 (EDGES) --\n"
        f"F1-Score        :  {metrics_history['val_f1_1'][best_idx]:.4f}\n"
        f"Precision       :  {metrics_history['val_precision_1'][best_idx]:.4f}\n"
        f"Recall          :  {metrics_history['val_recall_1'][best_idx]:.4f}\n\n"
        f"-- CLASS 0 (NON-EDGES) --\n"
        f"F1-Score        :  {metrics_history['val_f1_0'][best_idx]:.4f}\n"
        f"Precision       :  {metrics_history['val_precision_0'][best_idx]:.4f}\n"
        f"Recall          :  {metrics_history['val_recall_0'][best_idx]:.4f}\n"
    )

    axs[4, 1].text(0.5, 0.5, summary_text, fontsize=14, va='center', ha='center', family='monospace', 
                bbox=dict(facecolor="#f0f0f0", edgecolor='gray', alpha=0.8, boxstyle='round,pad=1.5'))


    plt.tight_layout()
    plt.show()

In [14]:
def normalize_data_inplace(X, mean=None, std=None, opt='flat'):
    if opt == 'desc':
        X = X.view(X.size(0), 20, 16).mean(dim=2)
    elif opt == 'flat':
        X = X.view(X.size(0), -1)
    else:
        raise ValueError("opt must be 'desc' or 'flat'")

    if mean is None or std is None:
        mean = X.mean(dim=0, keepdim=True)
        std = X.std(dim=0, keepdim=True) + 1e-8

    # In-place normalization
    X.sub_(mean).div_(std)

    return X, mean, std

In [15]:
def generate_all_datasets(train_pts, train_lb, train_ids):
    # Ensure directory exists
    if not Path('../data/processed_data').exists(): 
        print(f"Error: .npz datasets folder ../data/processed_data not found.")
        return None

    print("--- Computing full Train stats (std+mean) ---")
    if not (Path('../data/processed_data/global_train_mean.pt').exists() and Path('../data/processed_data/global_train_std.pt').exists()):
        X_train_full_raw, _ = load_full_dataset('Train') 
        X_train_full_tensor = torch.tensor(X_train_full_raw, dtype=torch.float)
        
        _, global_mean, global_std = normalize_data_inplace(X_train_full_tensor, opt='flat')
        torch.save(global_mean, '../data/processed_data/global_train_mean.pt')
        torch.save(global_std, '../data/processed_data/global_train_std.pt')
        
        del X_train_full_raw, X_train_full_t
        torch.cuda.empty_cache()
        print("Full Train stats saved.")
    else:
        print("Full Train stats already exist. Skipping.")

    print("\n--- Starting all downsamplings ---")
    for method, params_list in DOWNSAMPLING_GRID.items():
        for params in params_list:
            # Create an identity filename (ex: ds_fps_retention_rate-0.1.npz)
            param_str = "_".join([f"{k}-{v}" for k, v in params.items()])
            filename = f"../data/processed_data/ds_{method}_{param_str}.npz"
            
            # Safety check to not redo an alr done treatement
            if Path(filename).exists():
                print(f"✅ Skipping {filename} (Already computed)")
                continue
                
            print(f"⏳ Processing: {method} with {params} ...")
            X_ds_list, y_ds_list = build_downsampled_features(
                points_list=train_pts, labels_list=train_lb, file_ids=train_ids, 
                method=method, **params
            )
            
            X_ds = np.vstack(X_ds_list)
            y_ds = np.concatenate(y_ds_list)
            
            # Save compressed
            np.savez_compressed(filename, X_train=X_ds, y_train=y_ds)
            print(f"💾 Saved {filename} | Shape: {X_ds.shape}")

In [16]:
# Generate all datasets and save them to the disk
train_pts, train_lb, train_ids = load_dataset_ply_lb(dataset='Train')
generate_all_datasets(train_pts, train_lb, train_ids)

Loading : 20/198 files...
Loading : 40/198 files...
Loading : 60/198 files...
Loading : 80/198 files...
Loading : 100/198 files...
Loading : 120/198 files...
Loading : 140/198 files...
Loading : 160/198 files...
Loading : 180/198 files...
Successfully loaded 198/198 files.
--- Computing full Train stats (std+mean) ---
Full Train stats already exist. Skipping.

--- Starting all downsamplings ---
✅ Skipping ../data/processed_data/ds_voxel_resolution_percentage-0.01.npz (Already computed)
✅ Skipping ../data/processed_data/ds_voxel_resolution_percentage-0.02.npz (Already computed)
✅ Skipping ../data/processed_data/ds_voxel_resolution_percentage-0.03.npz (Already computed)
✅ Skipping ../data/processed_data/ds_voxel_resolution_percentage-0.04.npz (Already computed)
✅ Skipping ../data/processed_data/ds_voxel_resolution_percentage-0.05.npz (Already computed)
✅ Skipping ../data/processed_data/ds_fps_retention_rate-0.05.npz (Already computed)
✅ Skipping ../data/processed_data/ds_fps_retention_ra

In [17]:
def run_grid_search(X_val_raw, y_val_raw, device='cuda'):
    
    global_train_mean = torch.load('../data/processed_data/global_train_mean.pt')
    global_train_std = torch.load('../data/processed_data/global_train_std.pt')
    
    results_file = '../data/results/grid_search_metrics.csv'
    all_results = []
    completed_runs = set()

    # Safety check for crashes while training to not redo the same training if already donme before
    if Path(results_file).exists():
        print(f"Found existing results file at {results_file}. Parsing to check completed runs...")
        df_existing = pd.read_csv(results_file)
        
        # Build a set of unique signatures for already completed runs
        for _, row in df_existing.iterrows():
            s = f"{row['method']}_{row['ds_param']}_{row['norm_strategy']}_{row['learning_rate']}_{row['batch_size']}"
            completed_runs.add(s)
            
        # Load the existing data back into our list so we append to it not overwrite
        all_results = df_existing.to_dict('records')
        print(f"Resuming experiments. {len(completed_runs)} runs already completed.")
    else:
        print("No existing results found. Starting fresh.")

    # get the list of datasets 
    dataset_files = list(Path('../data/processed_data').glob('ds_*.npz'))    

    total_runs = len(dataset_files) * len(NORM_STRATEGIES) * len(LEARNING_RATES) * len(BATCH_SIZES)
    current_run = 0

    for ds_file in dataset_files:
        file_parts = ds_file.stem.split('_')
        method = file_parts[1]
        param_str = "_".join(file_parts[2:])
        
        # Use 'with' to auto-close the .npz file 
        with np.load(ds_file) as data:
            X_train_raw = data['X_train']
            y_train_raw = data['y_train']
        
        for norm in NORM_STRATEGIES:
            for lr in LEARNING_RATES:
                for bs in BATCH_SIZES:
                    current_run += 1
                    
                    # Create the signature for the current loop
                    current_s = f"{method}_{param_str}_{norm}_{lr}_{bs}"
                    
                    # Skip if alr done
                    if current_s in completed_runs:
                        print(f"[{current_run}/{total_runs}] ⏭️ Skipping (Done): {method} | {param_str} | Norm: {norm} | LR: {lr} | BS: {bs}")
                        continue
                        
                    print(f"\n[{current_run}/{total_runs}] 🚀 Running: {method} | {param_str} | Norm: {norm} | LR: {lr} | BS: {bs}")
                    
                    X_train_tensor = torch.tensor(X_train_raw, dtype=torch.float)
                    y_train_tensor = torch.tensor(y_train_raw, dtype=torch.long)
                    X_val_tensor = torch.tensor(X_val_raw, dtype=torch.float)
                    y_val_tensor = torch.tensor(y_val_raw, dtype=torch.long)
                    
                    if norm == 'val':
                        X_val_tensor, mean, std = normalize_data_inplace(X_val_tensor, opt='flat')
                        X_train_tensor, _, _ = normalize_data_inplace(X_train_tensor, mean=mean, std=std, opt='flat')
                    elif norm == 'train_ds':
                        X_train_tensor, mean, std = normalize_data_inplace(X_train_tensor, opt='flat')
                        X_val_tensor, _, _ = normalize_data_inplace(X_val_tensor, mean=mean, std=std, opt='flat')
                    elif norm == 'separate':
                        X_train_tensor, _, _ = normalize_data_inplace(X_train_tensor, opt='flat')
                        X_val_tensor, _, _ = normalize_data_inplace(X_val_tensor, opt='flat')
                    elif norm == 'train_full':
                        X_train_tensor, _, _ = normalize_data_inplace(X_train_tensor, mean=global_train_mean, std=global_train_std, opt='flat')
                        X_val_tensor, _, _ = normalize_data_inplace(X_val_tensor, mean=global_train_mean, std=global_train_std, opt='flat')
                        
                    #model = MLP(input_dim=320)
                    history, _ = train_mlp_model(
                        X_train_tensor, y_train_tensor, X_val_tensor, y_val_tensor,
                        nb_epochs=50, batch_size=bs, lr=lr, 
                        device=device, display_metrics=False, patience=10,
                    )
                    
                    best_idx = history['best_epoch'] - 1
                    
                    run_record = {
                        'method': method,
                        'ds_param': param_str,
                        'norm_strategy': norm,
                        'learning_rate': lr,
                        'batch_size': bs,
                        'best_epoch': history['best_epoch'],
                        'val_loss': history['val_loss'][best_idx],
                        'f1_class_1': history['val_f1_1'][best_idx],
                        'f1_class_0': history['val_f1_0'][best_idx],
                        'precision_1': history['val_precision_1'][best_idx],
                        'recall_1': history['val_recall_1'][best_idx],
                        'mcc': history['val_mcc'][best_idx],
                        'false_positives': history['val_fp'][best_idx],
                        'false_negatives': history['val_fn'][best_idx]
                    }
                    
                    all_results.append(run_record)
                    completed_runs.add(current_s)
                    
                    # Overwrite the CSV at everytime so nothing is lost if it crashes
                    df = pd.DataFrame(all_results)
                    df.to_csv(results_file, index=False)
                    
                    # memory cleanup 
                    del X_train_t, y_train_t, X_val_t, y_val_t, history
                    torch.cuda.empty_cache()
                    gc.collect()

In [18]:
# Run tests on all generated datasets
X_val_raw, y_val_raw = load_full_dataset('Validation')
run_grid_search(X_val_raw, y_val_raw)

Loading : 20/50 files...
Loading : 40/50 files...
Found existing results file at ../data/results/grid_search_metrics.csv. Parsing to check completed runs...
Resuming experiments. 408 runs already completed.
[1/408] ⏭️ Skipping (Done): fps | retention_rate-0.05 | Norm: val | LR: 0.01 | BS: 512
[2/408] ⏭️ Skipping (Done): fps | retention_rate-0.05 | Norm: val | LR: 0.01 | BS: 1024
[3/408] ⏭️ Skipping (Done): fps | retention_rate-0.05 | Norm: val | LR: 0.005 | BS: 512
[4/408] ⏭️ Skipping (Done): fps | retention_rate-0.05 | Norm: val | LR: 0.005 | BS: 1024
[5/408] ⏭️ Skipping (Done): fps | retention_rate-0.05 | Norm: val | LR: 0.001 | BS: 512
[6/408] ⏭️ Skipping (Done): fps | retention_rate-0.05 | Norm: val | LR: 0.001 | BS: 1024
[7/408] ⏭️ Skipping (Done): fps | retention_rate-0.05 | Norm: train_ds | LR: 0.01 | BS: 512
[8/408] ⏭️ Skipping (Done): fps | retention_rate-0.05 | Norm: train_ds | LR: 0.01 | BS: 1024
[9/408] ⏭️ Skipping (Done): fps | retention_rate-0.05 | Norm: train_ds | LR: 0.

In [ ]:
df = pd.read_csv('../data/results/grid_search_metrics.csv')

# Find the top 5 models with the highest F1-Score for edges
best_models = df.sort_values(by='f1_class_1', ascending=False).head(5)
display(best_models)

,method,ds_param,norm_strategy,learning_rate,batch_size,best_epoch,val_loss,f1_class_1,f1_class_0,precision_1,recall_1,mcc,false_positives,false_negatives
316,voxel,resolution_percentage-0.02,val,0.001,512,10,0.007724,0.989875,0.999375,0.988435,0.991319,0.989251,465,348
239,random,retention_rate-0.2,train_full,0.001,1024,13,0.008049,0.988779,0.999309,0.989532,0.988027,0.988088,419,480
335,voxel,resolution_percentage-0.02,train_full,0.001,1024,13,0.006248,0.988581,0.999296,0.989186,0.987977,0.987878,433,482
317,voxel,resolution_percentage-0.02,val,0.001,1024,7,0.005815,0.988064,0.999261,0.983946,0.992217,0.987335,649,312
221,random,retention_rate-0.2,val,0.001,1024,8,0.005947,0.987973,0.999259,0.988318,0.987628,0.987231,468,496
